In [ ]:
# This notebook demonstrates the preprocessing and feature extraction pipeline
# used in the study: Efficient Olive Leaf Disease Detection Using Deep Features,
# Composite Filter-Based Feature Selection, and Ensemble Learning.



import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet101, MobileNet
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet import preprocess_input as mobilenet_preprocess
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os


# Function to read the image and convert it to RGBA

def read_image_to_rgba(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    img_rgba = cv2.cvtColor(img, cv2.COLOR_BGR2RGBA)
    return img_rgba

# Function to create a color-based segmentation mask

def create_color_based_mask(img_rgba):
    white_mask = np.all(img_rgba[:, :, :3] > 100, axis=2)
    blue_mask = (img_rgba[:, :, 0] < 100) & (img_rgba[:, :, 1] < 100) & (img_rgba[:, :, 2] > 100)
    combined_mask = white_mask | blue_mask
    return combined_mask


# Function to fill holes using morphological operations

def fill_holes(mask):
    kernel = np.ones((5, 5), np.uint8)
    filled_mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    return filled_mask


# Function to apply the mask and remove background details

def apply_mask(img_rgba, mask):
    img_rgba[:, :, 3] = np.where(mask, 0, 255)  # Maskenin olduğu yerlerde alpha kanalını 0 yap
    return img_rgba


# Function to split the image horizontally

def split_image_horizontally(img_rgba):
    height, width = img_rgba.shape[:2]
    left_half = img_rgba[:, :width//2]
    right_half = img_rgba[:, width//2:]
    return left_half, right_half


# Feature extraction function
def extract_features(img, model, preprocess_input):
    img_resized = cv2.resize(img, (224, 224))
    img_array = image.img_to_array(img_resized)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    features = model.predict(img_array)
    return features.flatten()


# Main processing function
def process_and_extract_features(image_path, model, preprocess_input):
    img_rgba = read_image_to_rgba(image_path)
    mask = create_color_based_mask(img_rgba)
    filled_mask = fill_holes(mask)
    img_masked = apply_mask(img_rgba, filled_mask)
    #left_half, right_half = split_image_horizontally(img_masked)


    #left_features = extract_features(left_half, model, preprocess_input)
    #right_features = extract_features(right_half, model, preprocess_input)

    return extract_features(img_masked[:,:,0:3], model, preprocess_input)


def read_image_files(directory):
  image_files = []
  for filename in os.listdir(directory):
    if filename.endswith(".jpg") or filename.endswith(".png"):
      image_files.append(os.path.join(directory, filename))
  return image_files


# Sample usage
non_healthy_imgs = read_image_files("/gdrive/MyDrive/Colab Notebooks/olive_data/hastalıklı")
healthy_imgs = read_image_files("/gdrive/MyDrive/Colab Notebooks/olive_data/sağlam")
labels = [0, 1]  # label for each image (0: healty, 1: diseased)

resnet_model = ResNet101(weights='imagenet', include_top=False, pooling='avg')
mobilenet_model = MobileNet(weights='imagenet', include_top=False, pooling='avg')

features_non_healthy = []
features_non_healthy_res = []
features_healthy = []
features_healthy_res = []

for image_path in non_healthy_imgs:
    image_features_resnet = process_and_extract_features(image_path, resnet_model, resnet_preprocess)
    image_features_mobilenet = process_and_extract_features(image_path, mobilenet_model, mobilenet_preprocess)
    features_non_healthy.append(image_features_mobilenet)
    features_non_healthy_res.append(image_features_resnet)

for image_path in healthy_imgs:
    image_features_resnet = process_and_extract_features(image_path, resnet_model, resnet_preprocess)
    image_features_mobilenet = process_and_extract_features(image_path, mobilenet_model, mobilenet_preprocess)
    features_healthy.append(image_features_mobilenet)
    features_healthy_res.append(image_features_resnet)

#features = np.array(features)
#classify_features(features, labels)


/tmp/ipython-input-1418620786.py:84: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobilenet_model = MobileNet(weights='imagenet', include_top=False, pooling='avg')


1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 20s 20s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━

In [ ]:
all_features = features_non_healthy + features_healthy
all_features = np.array(all_features)

all_features_res = features_non_healthy_res + features_healthy_res
all_features_res = np.array(all_features_res)

all_labels = [0] * len(features_non_healthy) + [1] * len(features_healthy)
all_labels = np.array(all_labels)

In [ ]:

from sklearn.feature_selection import mutual_info_classif, f_classif, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def select_top_50_percent_features_all_methods(X, y, top_k_ratio = 0.5):
    """
    MI, F-score, Chi2, Hybrid Voting, Score Fusion, RF Importance, Intersect ve Union yöntemleriyle
    %50 en iyi özellikleri seçer.

    Returns:
        dict: Yöntem adına karşılık gelen seçilen feature indeksleri listesi
    """
    n_features = X.shape[1]
    top_k = int(X.shape[1] * top_k_ratio)


    # 1. Mutual Information
    mi = mutual_info_classif(X, y)
    mi_indices = np.argsort(mi)[-top_k:]

    # 2. F-score
    f_scores, _ = f_classif(X, y)
    f_indices = np.argsort(f_scores)[-top_k:]

    # 3. Chi2
    chi_scores, _ = chi2(X, y)
    chi_indices = np.argsort(chi_scores)[-top_k:]

# 4. Hybrid (majority voting – features appearing in at least two methods)
    votes = np.zeros(n_features)
    votes[mi_indices] += 1
    votes[f_indices] += 1
    votes[chi_indices] += 1
    hybrid_indices = np.where(votes >= 2)[0]
    if len(hybrid_indices) > top_k:
        hybrid_scores = mi[hybrid_indices] + f_scores[hybrid_indices] + chi_scores[hybrid_indices]
        hybrid_indices = hybrid_indices[np.argsort(hybrid_scores)[-top_k:]]

    # 5. Score Fusion (normalize and sum)
    scaler = MinMaxScaler()
    mi_norm = scaler.fit_transform(mi.reshape(-1, 1)).ravel()
    f_norm = scaler.fit_transform(f_scores.reshape(-1, 1)).ravel()
    chi_norm = scaler.fit_transform(chi_scores.reshape(-1, 1)).ravel()
    fusion_score = mi_norm + f_norm + chi_norm
    fusion_indices = np.argsort(fusion_score)[-top_k:]

    # 6. Intersect (common feaatures)
    intersect_indices = set(mi_indices).intersection(f_indices).intersection(chi_indices)
    intersect_indices = np.array(sorted(intersect_indices))
    if len(intersect_indices) > top_k:
        combined_scores = mi[intersect_indices] + f_scores[intersect_indices] + chi_scores[intersect_indices]
        intersect_indices = intersect_indices[np.argsort(combined_scores)[-top_k:]]

   # 7. Union (largest feature set)
    union_indices = set(mi_indices).union(f_indices).union(chi_indices)
    union_indices = np.array(sorted(union_indices))
    if len(union_indices) > top_k:
        combined_scores = mi[union_indices] + f_scores[union_indices] + chi_scores[union_indices]
        union_indices = union_indices[np.argsort(combined_scores)[-top_k:]]

    return {
        "mutual_info": mi_indices,
        "f_score": f_indices,
        "chi2": chi_indices,
        "hybrid": hybrid_indices,
        "score_fusion": fusion_indices,
        "intersect": intersect_indices,
        "union": union_indices
    }

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import numpy as np
import time

def train_models_without_feature_selection(X_dict, y):
    """
    - Without feature selection:
    - kNN and SVM training
    - Prediction collection with 10-fold CV - Calculation of accuracy, F1 and MCC scores

    Returns:
    predictions_dict: {model_name: predictions}
    metrics_dict: {model_name: {"accuracy":..., "f1":..., "mcc":...}}
    """
    predictions = {}
    metrics = {}

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    for set_name, X in X_dict.items():
        for clf_name, clf in {
            "knn": KNeighborsClassifier(n_neighbors=1),
            "svm": SVC(kernel="poly", degree=3),
            "lgb" :LGBMClassifier()
        }.items():
            all_preds = np.zeros_like(y)
            fold_accuracies = []
            fold_f1_scores = []
            fold_mcc_scores = []

            start_time = time.time()
            for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                clf.fit(X[train_idx], y[train_idx])
                preds = clf.predict(X[test_idx])
                all_preds[test_idx] = preds

                # Collect scores for each fold
                fold_accuracies.append(accuracy_score(y[test_idx], preds))
                fold_f1_scores.append(f1_score(y[test_idx], preds, average='binary'))
                fold_mcc_scores.append(matthews_corrcoef(y[test_idx], preds))
            end_time = time.time()

            key = f"{set_name}_allfeatures_{clf_name}"
            predictions[key] = all_preds

            # Performance metrics
            acc_mean = np.mean(fold_accuracies)
            acc_std = np.std(fold_accuracies)
            f1_mean = np.mean(fold_f1_scores)
            f1_std = np.std(fold_f1_scores)
            mcc_mean = np.mean(fold_mcc_scores)
            mcc_std = np.std(fold_mcc_scores)

            metrics[key] = {
                "accuracy_mean": acc_mean,
                "accuracy_std": acc_std,
                "f1_mean": f1_mean,
                "f1_std": f1_std,
                "mcc_mean": mcc_mean,
                "mcc_std": mcc_std,
                "training_time_seconds": end_time - start_time
            }
    return predictions, metrics

X_dict = {
    "mobilenet": all_features,
    "resnet": all_features_res,

}

# get predictions
predictions_dict, accuracies_dict= train_models_without_feature_selection(X_dict, all_labels)

[LightGBM] [Info] Number of positive: 515, number of negative: 343
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033772 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 209568
[LightGBM] [Info] Number of data points in the train set: 858, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.600233 -> initscore=0.406436
[LightGBM] [Info] Start training from score 0.406436
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 515, number of negative: 343
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023627 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 209477
[LightGBM] [Info] Number of data points in the train set: 858, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.600233 -> initscore=0.406436
[LightGBM] [Info] Start training from score 0.406436
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 514, number of negative: 344
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023775 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 209748
[LightGBM] [Info] Number of data points in the train set: 858, number of used features: 1024
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.599068 -> initscore=0.401582
[LightGBM] [Info] Start training from score 0.401582
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 515, number of negative: 344
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040425 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 453842
[LightGBM] [Info] Number of data points in the train set: 859, number of used features: 2048
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.599534 -> initscore=0.403525
[LightGBM] [Info] Start training from score 0.403525
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 515, number of negative: 344
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062249 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 453810
[LightGBM] [Info] Number of data points in the train set: 859, number of used features: 2048
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.599534 -> initscore=0.403525
[LightGBM] [Info] Start training from score 0.403525
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
accuracies_dict

{'mobilenet_allfeatures_knn': {'accuracy_mean': np.float64(0.9391447368421053),
  'accuracy_std': np.float64(0.023476386766514073),
  'f1_mean': np.float64(0.9516140872540669),
  'f1_std': np.float64(0.018071540081108537),
  'mcc_mean': np.float64(0.875779179493114),
  'mcc_std': np.float64(0.04749672902915247),
  'training_time_seconds': 0.1937112808227539},
 'mobilenet_allfeatures_svm': {'accuracy_mean': np.float64(0.9852850877192983),
  'accuracy_std': np.float64(0.013470143706291482),
  'f1_mean': np.float64(0.9879004020257078),
  'f1_std': np.float64(0.011020051718217402),
  'mcc_mean': np.float64(0.9695949386707257),
  'mcc_std': np.float64(0.027867162190194666),
  'training_time_seconds': 0.7310328483581543},
 'mobilenet_allfeatures_lgb': {'accuracy_mean': np.float64(0.976875),
  'accuracy_std': np.float64(0.016853212854239734),
  'f1_mean': np.float64(0.9811955821307079),
  'f1_std': np.float64(0.01360339933975604),
  'mcc_mean': np.float64(0.9525397539529064),
  'mcc_std': np.

In [ ]:
!pip install imblearn
import numpy as np
from sklearn.feature_selection import mutual_info_classif, f_classif, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import time

def select_top_50_percent_features_all_methods_balanced(X, y, top_k_ratio=0.5):
    """
    Selects the top 50% of features using MI, F-score, Chi2, Hybrid Voting, Score Fusion, RF Importance, Intersect, Union,
    SMOTE, and Classwise-Intersect methods.

    Returns:
    dict: List of selected feature indexes corresponding to the method name
    """
    start_time = time.time()

    n_features = X.shape[1]
    top_k = 128 #int(X.shape[1] * top_k_ratio)
    scaler = MinMaxScaler()

    # 1. Mutual Information, F-score, Chi2
    mi = mutual_info_classif(X, y)
    f_scores, _ = f_classif(X, y)
    chi_scores, _ = chi2(X, y)


    """
    # 2. Resampling with SMOTE

    sm = SMOTE(random_state=42)
    X_smote, y_smote = sm.fit_resample(X, y)
    mi_smote = mutual_info_classif(X_smote, y_smote)
    f_smote, _ = f_classif(X_smote, y_smote)
    chi_smote, _ = chi2(X_smote, y_smote)
    """
    # 3. Class-wise score averaging
    def classwise_mean_scores(X, y, score_func):
        scores = np.zeros(X.shape[1])
        classes = np.unique(y)
        for cls in classes:
            X_cls = X[y == cls]
            y_cls = y[y == cls]
            scores += score_func(X_cls, y_cls)

        return scores / len(classes)

    mi_classwise = classwise_mean_scores(X, y, lambda Xc, yc: mutual_info_classif(Xc, yc))
    f_classwise = classwise_mean_scores(X, y, lambda Xc, yc: f_classif(Xc, yc)[0])
    chi_classwise = classwise_mean_scores(X, y, lambda Xc, yc: chi2(Xc, yc)[0])

    # 4. RF Importance
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X, y)
    rf_importance = rf.feature_importances_

    # 5. Hybrid (majority voting)
    votes = np.zeros(n_features)
    for arr in [mi, f_scores, chi_scores]:
        indices = np.argsort(arr)[-top_k:]
        votes[indices] += 1
    hybrid_indices = np.where(votes >= 2)[0]
    if len(hybrid_indices) > top_k:
        hybrid_scores = mi[hybrid_indices] + f_scores[hybrid_indices] + chi_scores[hybrid_indices]
        hybrid_indices = hybrid_indices[np.argsort(hybrid_scores)[-top_k:]]

    # 6. Score Fusion
    mi_norm = scaler.fit_transform(mi.reshape(-1, 1)).ravel()
    f_norm = scaler.fit_transform(f_scores.reshape(-1, 1)).ravel()
    chi_norm = scaler.fit_transform(chi_scores.reshape(-1, 1)).ravel()
    fusion_score = mi_norm + f_norm + chi_norm
    fusion_indices = np.argsort(fusion_score)[-top_k:]

    # 7. Intersect
    intersect_indices = set(np.argsort(mi)[-top_k:]).intersection(
                         np.argsort(f_scores)[-top_k:]).intersection(
                         np.argsort(chi_scores)[-top_k:])
    intersect_indices = np.array(sorted(intersect_indices))
    if len(intersect_indices) > top_k:
        combined_scores = mi[intersect_indices] + f_scores[intersect_indices] + chi_scores[intersect_indices]
        intersect_indices = intersect_indices[np.argsort(combined_scores)[-top_k:]]

    # 8. Union
    union_indices = set(np.argsort(mi)[-top_k:]).union(
                    np.argsort(f_scores)[-top_k:]).union(
                    np.argsort(chi_scores)[-top_k:])
    union_indices = np.array(sorted(union_indices))
    if len(union_indices) > top_k:
        combined_scores = mi[union_indices] + f_scores[union_indices] + chi_scores[union_indices]
        union_indices = union_indices[np.argsort(combined_scores)[-top_k:]]

    # 9. Intersect of classwise scores
    mi_cls_idx = np.argsort(mi_classwise)[-top_k:]
    f_cls_idx = np.argsort(f_classwise)[-top_k:]
    chi_cls_idx = np.argsort(chi_classwise)[-top_k:]
    intersect_classwise = set(mi_cls_idx).intersection(f_cls_idx).intersection(chi_cls_idx)
    intersect_classwise = np.array(sorted(intersect_classwise))
    if len(intersect_classwise) > top_k:
        combined_cls_scores = mi_classwise[intersect_classwise] + f_classwise[intersect_classwise] + chi_classwise[intersect_classwise]
        intersect_classwise = intersect_classwise[np.argsort(combined_cls_scores)[-top_k:]]

    # Helper function
    def top_k_indices(arr): return np.argsort(arr)[-top_k:]

    end_time = time.time()
    print(f"Feature selection running time: {end_time - start_time:.4f} seconds")

    return {
        "mutual_info": top_k_indices(mi),
        "f_score": top_k_indices(f_scores),
        "chi2": top_k_indices(chi_scores),
        #"mi_smote": top_k_indices(mi_smote),
        #"f_smote": top_k_indices(f_smote),
        #"chi_smote": top_k_indices(chi_smote),
        "mi_classwise": top_k_indices(mi_classwise),
        "f_classwise": top_k_indices(f_classwise),
        "chi_classwise": top_k_indices(chi_classwise),
        #"rf_importance": top_k_indices(rf_importance),
        "hybrid": hybrid_indices,
        "score_fusion": fusion_indices,
        "intersect": intersect_indices,
        "union": union_indices
        #"intersect_classwise": intersect_classwise
    }

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import numpy as np
import time

def train_models_and_collect_predictions_with_scores(X_dict, y):
    """
    - For all dataset and method combinations:
    - 50% feature selection (MI, F-score, Chi2, etc.)
    - kNN and SVM training
    - Prediction collection with 10-fold CV - Accuracy calculation - Recording used feature indexes

    Returns:
    predictions_dict: {model_name: predictions}
    accuracies_dict: {model_name: accuracy}
    selected_features_dict: {model_name: used feature indexes}
    """
    predictions = {}
    metrics = {}
    selected_features = {}

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for set_name, X in X_dict.items():
        start_time_fs = time.time()
        selected_indices_dict = select_top_50_percent_features_all_methods_balanced(X, y, 0.1)
        end_time_fs = time.time()
        print(f"Feature selection for {set_name} took {end_time_fs - start_time_fs:.4f} seconds")

        for method_name, indices in selected_indices_dict.items():
            X_sel = X[:, indices]

            for clf_name, clf in {
                "knn": KNeighborsClassifier(n_neighbors=1),
                "svm": SVC(kernel="poly", degree=3),
                "lgb" :LGBMClassifier()

            }.items():
                all_preds = np.zeros_like(y)
                fold_accuracies = []
                fold_f1_scores = []
                fold_mcc_scores = []

                start_time_clf = time.time()
                for train_idx, test_idx in skf.split(X_sel, y):
                    clf.fit(X_sel[train_idx], y[train_idx])
                    preds = clf.predict(X_sel[test_idx])
                    all_preds[test_idx] = preds

                    fold_accuracies.append(accuracy_score(y[test_idx], preds))
                    fold_f1_scores.append(f1_score(y[test_idx], preds, average='binary'))
                    fold_mcc_scores.append(matthews_corrcoef(y[test_idx], preds))
                end_time_clf = time.time()

                key = f"{set_name}_{method_name}_{clf_name}"
                predictions[key] = all_preds

                acc_mean = np.mean(fold_accuracies)
                acc_std = np.std(fold_accuracies)
                f1_mean = np.mean(fold_f1_scores)
                f1_std = np.std(fold_f1_scores)
                mcc_mean = np.mean(fold_mcc_scores)
                mcc_std = np.std(fold_mcc_scores)

                metrics[key] = {
                    "accuracy_mean": acc_mean,
                    "accuracy_std": acc_std,
                    "f1_mean": f1_mean,
                    "f1_std": f1_std,
                    "mcc_mean": mcc_mean,
                    "mcc_std": mcc_std,
                    "training_time_seconds": end_time_clf - start_time_clf
                }
                selected_features[key] = indices  # Özellik indekslerini kaydet

    return predictions, metrics, selected_features

In [ ]:

# get predictions.
predictions_dict, accuracies_dict, selected_features_dict = train_models_and_collect_predictions_with_scores(X_dict, all_labels)

# Example:
print("Best model:", max(accuracies_dict.items(), key=lambda x: x[1]['accuracy_mean'])[0])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:107: RuntimeWarning: divide by zero encountered in divide
  msb = ssbn / float(dfbn)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:107: RuntimeWarning: invalid value encountered in divide
  msb = ssbn / float(dfbn)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:107: RuntimeWarning: divide by zero encountered in divide
  msb = ssbn / float(dfbn)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:107: RuntimeWarning: invalid value encountered in divide
  msb = ssbn / float(dfbn)
/tmp/ipython-input-105980754.py:43: RuntimeWarning: invalid value encountered in add
  scores += score_func(X_cls, y_cls)


Feature selection running time: 24.8782 seconds
Feature selection for mobilenet took 24.8788 seconds
[LightGBM] [Info] Number of positive: 458, number of negative: 305
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006956 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25387
[LightGBM] [Info] Number of data points in the train set: 763, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.600262 -> initscore=0.406557
[LightGBM] [Info] Start training from score 0.406557
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 458, number of negative: 305
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002745 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25451
[LightGBM] [Info] Number of data points in the train set: 763, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.600262 -> initscore=0.406557
[LightGBM] [Info] Start training from score 0.406557
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 457, number of negative: 306
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001454 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 30787
[LightGBM] [Info] Number of data points in the train set: 763, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.598952 -> initscore=0.401098
[LightGBM] [Info] Start training from score 0.401098
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 458, number of negative: 306
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001406 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 30813
[LightGBM] [Info] Number of data points in the train set: 764, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.599476 -> initscore=0.403284
[LightGBM] [Info] Start training from score 0.403284
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
En iyi model: resnet_f_score_svm


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# prompt: sort accuracies_dict descending based on accuracy

sorted_accuracies = sorted(accuracies_dict.items(), key=lambda item: item[1]['accuracy_mean'], reverse=True)
sorted_accuracies

[('resnet_f_score_svm',
  {'accuracy_mean': np.float64(0.9884651419123726),
   'accuracy_std': np.float64(0.009015415713015422),
   'f1_mean': np.float64(0.9904678803394802),
   'f1_std': np.float64(0.007395950107581889),
   'mcc_mean': np.float64(0.975990703680796),
   'mcc_std': np.float64(0.018770539334639347),
   'training_time_seconds': 0.09181094169616699}),
 ('resnet_hybrid_svm',
  {'accuracy_mean': np.float64(0.9874180214935244),
   'accuracy_std': np.float64(0.01078563177931908),
   'f1_mean': np.float64(0.9896315207048373),
   'f1_std': np.float64(0.008800498327014544),
   'mcc_mean': np.float64(0.973839790748978),
   'mcc_std': np.float64(0.022404210592183737),
   'training_time_seconds': 0.05963420867919922}),
 ('resnet_score_fusion_svm',
  {'accuracy_mean': np.float64(0.9874180214935244),
   'accuracy_std': np.float64(0.007842626747501077),
   'f1_mean': np.float64(0.9896310442386913),
   'f1_std': np.float64(0.0063481466614360126),
   'mcc_mean': np.float64(0.973958384508

In [ ]:
from scipy.stats import mode
from sklearn.metrics import accuracy_score
import numpy as np

def incremental_voting_from_top_models(predictions_dict, accuracies_dict, selected_features_dict, y_true, start_n=3):
    """
    Performs incremental voting starting from the best 'start_n' models.
    Returns the best voting combination and its associated attributes in dictionary format.

    Parameters:
    predictions_dict (dict): model_name -> prediction vector
    accuracies_dict (dict): model_name -> accuracy rate
    selected_features_dict (dict): model_name -> attribute indexes used
    y_true (array): true labels
    start_n (int): initial number of models

    Returns:
    best_models (list): model names in the voting set that provides the highest accuracy
    best_accuracy (float): highest accuracy
    features_dict (dict): attribute indexes used for each model name
    all_results (list): for each step (model list, accuracy)
    """
    sorted_models = sorted(accuracies_dict.items(), key=lambda x: x[1]['accuracy_mean'], reverse=True)

    model_names_ordered = [m[0] for m in sorted_models]

    best_accuracy = 0
    best_models = []
    features_dict = {}
    all_results = []
    metrics = {}
    for i in range(start_n, len(model_names_ordered) + 1):
        selected = model_names_ordered[:i]
        preds_matrix = np.array([predictions_dict[m] for m in selected])
        voted_preds, _ = mode(preds_matrix, axis=0)
        voted_preds = voted_preds.flatten()
        acc = accuracy_score(y_true, voted_preds)
        all_results.append((selected.copy(), acc))

        if acc > best_accuracy:
            best_accuracy = acc
            best_models = selected.copy()

            acc = accuracy_score(y_true, voted_preds)
            f1 = f1_score(y_true, voted_preds, average='binary')  # binary classification varsayımı
            mcc = matthews_corrcoef(y_true, voted_preds)

            metrics["best"] = {
                    "accuracy": acc,
                    "f1": f1,
                    "mcc": mcc
                }
            features_dict = {m: selected_features_dict[m] for m in best_models}

    return best_models, best_accuracy, features_dict, all_results, metrics

In [ ]:
start_time = time.time()

best_models, best_accuracy, features_per_model, all_voting_results, metrics = incremental_voting_from_top_models(
    predictions_dict,
    accuracies_dict,
    selected_features_dict,
    y_true=all_labels,
    start_n=3
)
end_time = time.time()
print("Running Time: ",end_time-start_time)


print("Model group that provides the best accuracy:", best_models)
print("Accuracy:", best_accuracy)
print("Selected features for each model")
for model, feats in features_per_model.items():
    print(f"{model} -> {feats[:10]}... (toplam: {len(feats)})")

for models_used, acc in all_voting_results:
    print(f"Models: {models_used} → Accuracy: {acc:.4f}")


Süre:  0.4906961917877197
📌 En iyi doğruluğu veren model grubu: ['resnet_f_score_svm', 'resnet_hybrid_svm', 'resnet_score_fusion_svm', 'resnet_mutual_info_svm', 'mobilenet_mutual_info_svm', 'resnet_mutual_info_lgb', 'resnet_chi2_svm', 'resnet_union_svm', 'mobilenet_hybrid_lgb', 'mobilenet_hybrid_svm', 'resnet_intersect_svm', 'mobilenet_union_lgb', 'mobilenet_union_svm', 'mobilenet_intersect_lgb']
✅ Doğruluk: 0.9905660377358491
📊 Model bazlı seçilen öznitelikler:
resnet_f_score_svm -> [ 515 1335  410  832  511  226 1278  302 1456  407]... (toplam: 128)
resnet_hybrid_svm -> [  0   2   5  38  64  75  77  83  94 103]... (toplam: 121)
resnet_score_fusion_svm -> [ 465 1007 1964   10  567 1800  483 1292  247  675]... (toplam: 128)
resnet_mutual_info_svm -> [ 238 1440  789  572 1050  410  569  959 1538  338]... (toplam: 128)
mobilenet_mutual_info_svm -> [590 840 149  45  30  41 139 679 804 357]... (toplam: 128)
resnet_mutual_info_lgb -> [ 238 1440  789  572 1050  410  569  959 1538  338]... (t

In [ ]:
metrics

{'best': {'accuracy': 0.9905660377358491,
  'f1': 0.9921807124239791,
  'mcc': np.float64(0.9804079615481)}}

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import mode
import random
####GENETIC ALGORTIHM IMPLEMENTATION###
# ----------------------------
# 1. Data Preparation
# 
# ----------------------------
np.random.seed(42)
true_labels = all_labels  # dışarıdan geliyor (1D np.array)

model_names = list(predictions_dict.keys())
predictions = list(predictions_dict.values())
n_classifiers = len(predictions)

# ----------------------------
# 2. Majority Voting
# ----------------------------
def majority_vote(preds_subset):
    preds_array = np.array(preds_subset)
    voted = mode(preds_array, axis=0).mode
    return voted.flatten()

# ----------------------------
# 3. Fitness Calculation
# ----------------------------
def fitness(individual, predictions, true_labels):
    selected_preds = [pred for i, pred in enumerate(predictions) if individual[i] == 1]
    if len(selected_preds) == 0:
        return 0
    voted = majority_vote(selected_preds)
    return accuracy_score(true_labels, voted)

# ----------------------------
# 4. Tournament Selection

# ----------------------------
def tournament_selection(population, fitness_scores, k=5):
    selected_indices = np.random.choice(len(population), k, replace=False)
    selected = [(population[i], fitness_scores[i]) for i in selected_indices]
    selected.sort(key=lambda x: x[1], reverse=True)
    return selected[0][0]

# ----------------------------
# 5. Uniform Crossover
# ----------------------------
def uniform_crossover(parent1, parent2):
    return np.array([p1 if np.random.rand() > 0.5 else p2 for p1, p2 in zip(parent1, parent2)])

# ----------------------------
# 6. Advanced Genetic Algorithm
# ----------------------------
def genetic_algorithm_voting_advanced(predictions, true_labels, population_size=75, generations=150, mutation_rate=0.3):
    n_classifiers = len(predictions)
    population = [np.random.randint(0, 2, n_classifiers) for _ in range(population_size)]

    for generation in range(generations):
        print(f"Generation {generation + 1}/{generations}")
        fitness_scores = [fitness(ind, predictions, true_labels) for ind in population]
        elite_idx = np.argmax(fitness_scores)
        new_population = [population[elite_idx]]  # Elit birey

        while len(new_population) < population_size:
            parent1 = tournament_selection(population, fitness_scores)
            parent2 = tournament_selection(population, fitness_scores)
            child = uniform_crossover(parent1, parent2)

            for i in range(n_classifiers):
                if np.random.rand() < mutation_rate:
                    child[i] = 1 - child[i]

            new_population.append(child)

        population = new_population

    final_fitness = [fitness(ind, predictions, true_labels) for ind in population]
    best_idx = np.argmax(final_fitness)
    return population[best_idx], final_fitness[best_idx]

# ----------------------------
# 7. Apply and Print Results
# ----------------------------
start_time = time.time()
best_subset, best_score = genetic_algorithm_voting_advanced(predictions, true_labels)
end_time = time.time()
selected_model_names = [model_names[i] for i in range(n_classifiers) if best_subset[i] == 1]

print("Running Time: ",end_time-start_time)
print("\n The best model combination selected:")
for name in selected_model_names:
    print(f"✅ {name}")
print(f"\n🎯  Obtained accuracy with GA: {round(best_score * 100, 2)}%")

selected_preds = [predictions[i] for i in range(n_classifiers) if best_subset[i] == 1]
final_prediction = majority_vote(selected_preds)

print("\n Classification Performance:\n")
print(classification_report(true_labels, final_prediction))


Generation 1/150
Generation 2/150
Generation 3/150
Generation 4/150
Generation 5/150
Generation 6/150
Generation 7/150
Generation 8/150
Generation 9/150
Generation 10/150
Generation 11/150
Generation 12/150
Generation 13/150
Generation 14/150
Generation 15/150
Generation 16/150
Generation 17/150
Generation 18/150
Generation 19/150
Generation 20/150
Generation 21/150
Generation 22/150
Generation 23/150
Generation 24/150
Generation 25/150
Generation 26/150
Generation 27/150
Generation 28/150
Generation 29/150
Generation 30/150
Generation 31/150
Generation 32/150
Generation 33/150
Generation 34/150
Generation 35/150
Generation 36/150
Generation 37/150
Generation 38/150
Generation 39/150
Generation 40/150
Generation 41/150
Generation 42/150
Generation 43/150
Generation 44/150
Generation 45/150
Generation 46/150
Generation 47/150
Generation 48/150
Generation 49/150
Generation 50/150
Generation 51/150
Generation 52/150
Generation 53/150
Generation 54/150
Generation 55/150
Generation 56/150
G

In [ ]:
print(accuracy_score(true_labels, final_prediction))  # binary classification 
print(f1_score(true_labels, final_prediction, average='binary'))  # binary classification 
print(matthews_corrcoef(true_labels, final_prediction))

0.989517819706499
0.991304347826087
0.9781969933445143


In [ ]:
!pip install bayesian-optimization

In [ ]:
from bayes_opt import BayesianOptimization
import numpy as np
from sklearn.metrics import accuracy_score

# ----------------------------
# Example: prediction_dict -> each contains softmax predictions (n_samples x n_classes)
# ----------------------------
model_names = list(predictions_dict.keys())
predictions = list(predictions_dict.values())  # her biri: np.array, shape = (n_samples, n_classes)
true_labels = all_labels  # 1D np.array

n_models = len(predictions)
n_classes = 2



# predictions: her biri 1D array (n_samples,) -> class labels
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
encoder.fit(true_labels.reshape(-1, 1))  # sınıf sayısını belirle

# Convert to one-hot encoding
predictions_onehot = [encoder.transform(pred.reshape(-1, 1)) for pred in predictions]

# Weighted vote function
def weighted_vote(weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)  # normalize
    combined = np.tensordot(weights, predictions_onehot, axes=(0, 0))  # weighted sum
    final_pred = np.argmax(combined, axis=1)
    return final_pred


# ----------------------------
# Objective Function
# ----------------------------
def bayes_objective(**weight_dict):
    weights = [weight_dict[f'w{i}'] for i in range(n_models)]
    final_pred = weighted_vote(weights)
    acc = accuracy_score(true_labels, final_pred)
    return acc

# ----------------------------
# Parameter Range Definition
# ----------------------------
pbounds = {f'w{i}': (0, 1) for i in range(n_models)}

# ----------------------------
# Bayes Optimization
# ----------------------------
optimizer = BayesianOptimization(
    f=bayes_objective,
    pbounds=pbounds,
    random_state=42,
    verbose=2,
)
start_time = time.time()
optimizer.maximize(init_points=15, n_iter=30)

# ----------------------------
# Voting and Reporting with Best Weights
# ----------------------------
best_weights = [optimizer.max['params'][f'w{i}'] for i in range(n_models)]
best_weights = np.array(best_weights) / np.sum(best_weights)  # normalize
end_time = time.time()
print("Running Time: ",end_time-start_time)

print("\n Best weights:")
for name, weight in zip(model_names, best_weights):
    print(f"{name}: {round(weight, 4)}")

final_preds = weighted_vote(best_weights)

from sklearn.metrics import classification_report
print("\n Classification Report:\n")
print(classification_report(true_labels, final_preds))

|   iter    |  target   |    w0     |    w1     |    w2     |    w3     |    w4     |    w5     |    w6     |    w7     |    w8     |    w9     |    w10    |    w11    |    w12    |    w13    |    w14    |    w15    |    w16    |    w17    |    w18    |    w19    |    w20    |    w21    |    w22    |    w23    |    w24    |    w25    |    w26    |    w27    |    w28    |    w29    |    w30    |    w31    |    w32    |    w33    |    w34    |    w35    |    w36    |    w37    |    w38    |    w39    |    w40    |    w41    |    w42    |    w43    |    w44    |    w45    |    w46    |    w47    |    w48    |    w49    |    w50    |    w51    |    w52    |    w53    |    w54    |    w55    |    w56    |    w57    |    w58    |    w59    |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
print(accuracy_score(true_labels, final_preds))
print(matthews_corrcoef(true_labels, final_preds))
print(f1_score(true_labels, final_preds))


0.9853249475890985
0.9697090176257035
0.9879101899827288


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.preprocessing import OneHotEncoder
from bayes_opt import BayesianOptimization
import random

# ----------------------------
# Majority Voting Function
# ----------------------------
def majority_vote(preds_subset):
    preds_array = np.array(preds_subset)
    voted = mode(preds_array, axis=0).mode
    return voted.flatten()

# ----------------------------
# GA Fitness Function
# ----------------------------
def fitness(individual, predictions, true_labels):
    selected_preds = [pred for i, pred in enumerate(predictions) if individual[i] == 1]
    if len(selected_preds) == 0:
        return 0
    voted = majority_vote(selected_preds)
    return accuracy_score(true_labels, voted)

# ----------------------------
# Model selection with GA
# ----------------------------
def genetic_algorithm_selection(predictions, true_labels, population_size=150, generations=150, mutation_rate=0.15):
    n_classifiers = len(predictions)
    population = [np.random.randint(0, 2, n_classifiers) for _ in range(population_size)]

    for generation in range(generations):
        print(f"Generation {generation + 1}/{generations}")
        fitness_scores = [fitness(ind, predictions, true_labels) for ind in population]
        elite_idx = np.argmax(fitness_scores)
        new_population = [population[elite_idx]]

        while len(new_population) < population_size:
            parents = random.choices(population, k=2)
            crossover_point = np.random.randint(1, n_classifiers - 1)
            child = np.concatenate([parents[0][:crossover_point], parents[1][crossover_point:]])
            for i in range(n_classifiers):
                if np.random.rand() < mutation_rate:
                    child[i] = 1 - child[i]
            new_population.append(child)
        population = new_population

    final_fitness = [fitness(ind, predictions, true_labels) for ind in population]
    best_idx = np.argmax(final_fitness)
    return population[best_idx], final_fitness[best_idx]

# ----------------------------
# Weight Optimization with BO
# ----------------------------
def bayesian_weight_optimization(selected_preds, true_labels):
    encoder = OneHotEncoder(sparse_output=False)
    encoder.fit(true_labels.reshape(-1, 1))
    preds_onehot = [encoder.transform(p.reshape(-1, 1)) for p in selected_preds]
    n_models = len(preds_onehot)

    def weighted_vote(**kwargs):
        weights = np.array([kwargs[f'w{i}'] for i in range(n_models)])
        weights = weights / np.sum(weights)
        combined = np.tensordot(weights, preds_onehot, axes=(0, 0))
        final_pred = np.argmax(combined, axis=1)
        return accuracy_score(true_labels, final_pred)

    pbounds = {f'w{i}': (0, 1) for i in range(n_models)}
    optimizer = BayesianOptimization(f=weighted_vote, pbounds=pbounds, random_state=42)
    optimizer.maximize(init_points=15, n_iter=70)

    best_weights = np.array([optimizer.max['params'][f'w{i}'] for i in range(n_models)])
    best_weights /= best_weights.sum()
    return best_weights



def hybrid_pipeline(predictions_dict, true_labels):
    model_names = list(predictions_dict.keys())
    predictions = list(predictions_dict.values())

    # 1. Model selection using GA
    best_subset, best_score = genetic_algorithm_selection(predictions, true_labels)
    selected_model_names = [model_names[i] for i in range(len(best_subset)) if best_subset[i] == 1]
    selected_preds = [predictions[i] for i in range(len(best_subset)) if best_subset[i] == 1]

    # 2. Weight optimization using BO
    best_weights = bayesian_weight_optimization(selected_preds, true_labels)

    # 3. Final prediction with GA-BO ensemble 
    encoder = OneHotEncoder(sparse_output=False)
    encoder.fit(true_labels.reshape(-1, 1))
    preds_onehot = [encoder.transform(p.reshape(-1, 1)) for p in selected_preds]
    combined = np.tensordot(best_weights, preds_onehot, axes=(0, 0))
    final_prediction = np.argmax(combined, axis=1)

    return {
        'selected_models': selected_model_names,
        'weights': best_weights,
        'accuracy': accuracy_score(true_labels, final_prediction),
        'mcc': matthews_corrcoef(true_labels, final_prediction),
        'f1': f1_score(true_labels, final_prediction),
        'final_prediction': final_prediction
    }


model_names = list(predictions_dict.keys())
predictions = list(predictions_dict.values())  # her biri: np.array, shape = (n_samples, n_classes)
true_labels = all_labels  # 1D np.array

n_models = len(predictions)
n_classes = 2

start_time = time.time()
result = hybrid_pipeline(predictions_dict, true_labels)
end_time = time.time()
print("Running time: ",end_time-start_time)
print("Selected models:", result['selected_models'])
print("Number of models:", len(result['selected_models']))
print("Weigths:", result['weights'])
print("Accuracy:", result['accuracy'])
print("MCC:", result['mcc'])
print("F1:", result['f1'])


Generation 1/150
Generation 2/150
Generation 3/150
Generation 4/150
Generation 5/150
Generation 6/150
Generation 7/150
Generation 8/150
Generation 9/150
Generation 10/150
Generation 11/150
Generation 12/150
Generation 13/150
Generation 14/150
Generation 15/150
Generation 16/150
Generation 17/150
Generation 18/150
Generation 19/150
Generation 20/150
Generation 21/150
Generation 22/150
Generation 23/150
Generation 24/150
Generation 25/150
Generation 26/150
Generation 27/150
Generation 28/150
Generation 29/150
Generation 30/150
Generation 31/150
Generation 32/150
Generation 33/150
Generation 34/150
Generation 35/150
Generation 36/150
Generation 37/150
Generation 38/150
Generation 39/150
Generation 40/150
Generation 41/150
Generation 42/150
Generation 43/150
Generation 44/150
Generation 45/150
Generation 46/150
Generation 47/150
Generation 48/150
Generation 49/150
Generation 50/150
Generation 51/150
Generation 52/150
Generation 53/150
Generation 54/150
Generation 55/150
Generation 56/150
G